# Code Review

코드 리뷰를 위한 노트북입니다.


In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient

# #region agent log
with open('/Users/oasishun/workspace/n8n_study/.cursor/debug.log', 'a') as f:
    f.write(json.dumps({"sessionId": "debug-session", "runId": "run1", "hypothesisId": "A", "location": "code_review.ipynb:Cell1", "message": "Cell1 execution started", "data": {}, "timestamp": int(__import__('time').time() * 1000)}) + '\n')
# #endregion

# .env 파일에서 환경 변수 로드
load_dotenv()

# GitHub Personal Access Token 가져오기
github_token = os.getenv('GITHUB_TOKEN')

# OpenAI API Key 가져오기
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    print("⚠️ 경고: OPENAI_API_KEY가 설정되지 않았습니다. .env 파일에 OPENAI_API_KEY를 추가해주세요.")

# Slack 설정 가져오기
slack_bot_token = os.getenv('SLACK_BOT_TOKEN')
slack_team_id = os.getenv('SLACK_TEAM_ID')
slack_channel_ids = os.getenv('SLACK_CHANNEL_IDS')

# #region agent log
with open('/Users/oasishun/workspace/n8n_study/.cursor/debug.log', 'a') as f:
    f.write(json.dumps({"sessionId": "debug-session", "runId": "run1", "hypothesisId": "A", "location": "code_review.ipynb:Cell1", "message": "GitHub token loaded", "data": {"token_exists": github_token is not None}, "timestamp": int(__import__('time').time() * 1000)}) + '\n')
# #endregion

# GitHub MCP 서버 설정
mcp_config = {
    "github": {
        "url": "https://api.githubcopilot.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {github_token}"
        },
        "transport": "streamable_http"
    }
}

# Slack 설정이 있는 경우에만 추가
if slack_bot_token and slack_team_id and slack_channel_ids:
    mcp_config["slack"] = {
      "command": "docker",
      "args": [
        "run",
        "-i",
        "--rm",
        "-e",
        "SLACK_BOT_TOKEN",
        "-e",
        "SLACK_TEAM_ID",
        "-e",
        "SLACK_CHANNEL_IDS",
        "mcp/slack"
      ],
      "env": {
        "SLACK_BOT_TOKEN": slack_bot_token,
        "SLACK_TEAM_ID": slack_team_id,
        "SLACK_CHANNEL_IDS": slack_channel_ids
      },
      "transport": "stdio"
    }

    
# MCP 클라이언트 초기화
mcp_client = MultiServerMCPClient(mcp_config)
print(f"✅ GitHub MCP 클라이언트가 초기화되었습니다.")



/Users/oasishun/workspace/n8n_study/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ GitHub MCP 클라이언트가 초기화되었습니다.


In [2]:
tool_list = await mcp_client.get_tools()
print(f"✅ 도구 목록을 가져왔습니다. 총 {len(tool_list)}개의 도구가 있습니다.")

✅ 도구 목록을 가져왔습니다. 총 48개의 도구가 있습니다.


In [3]:
# langgraph.prebuilt에서 import (deprecation 경고는 무시)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# OpenAI 모델 초기화 (API 키 명시적 전달)
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY가 설정되지 않았습니다. .env 파일에 OPENAI_API_KEY를 추가해주세요.")

# 모델을 직접 초기화하여 API 키 전달
model = ChatOpenAI(
    model="gpt-4o",  # gpt-4.1은 존재하지 않으므로 gpt-4o로 변경
    api_key=openai_api_key,
    temperature=0
)

agent = create_react_agent(
    model=model,
    tools=tool_list,
    prompt="Use the tools provided to you to answer the user's question."
)

/var/folders/5f/q13b_2x93yg8tt704_qz66qr0000gn/T/ipykernel_96190/2996900392.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [4]:
async def process_stream(stream_generator):
    
    results = []
    try:
        async for chunk in stream_generator:
            key = list(chunk.keys())[0]
            if key == 'agent':
                # agent는 딕셔너리이거나 객체일 수 있음
                agent_data = chunk['agent']
                if isinstance(agent_data, dict):
                    messages = agent_data.get('messages', [])
                else:
                    messages = getattr(agent_data, 'messages', [])
                if messages:
                    content = getattr(messages[0], 'content', '') if hasattr(messages[0], 'content') else ''
            elif key == 'tools':
                # tools의 구조를 안전하게 처리
                tools_data = chunk['tools']
                
                # tools_data가 딕셔너리인 경우
                if isinstance(tools_data, dict):
                    messages = tools_data.get('messages', [])
                # tools_data가 직접 메시지 리스트인 경우
                elif isinstance(tools_data, list):
                    messages = tools_data
                # tools_data가 객체인 경우
                else:
                    messages = getattr(tools_data, 'messages', [])
                
                for tool_msg in messages:
                    # ToolMessage는 객체이므로 속성으로 접근
                    tool_name = getattr(tool_msg, 'name', 'Unknown')
                    tool_content = getattr(tool_msg, 'content', str(tool_msg))
                    print(f"Tool: {tool_name}")
                    print(f"Tool Output: {tool_content}")
                    print("-"*100)
            results.append(chunk)
        return results
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        # 디버깅을 위해 chunk 구조 출력
        if 'chunk' in locals():
            print(f"\nChunk structure: {type(chunk)}")
            print(f"Chunk keys: {chunk.keys() if hasattr(chunk, 'keys') else 'N/A'}")
            if key == 'tools':
                print(f"Tools data type: {type(chunk.get('tools', 'N/A'))}")
        return results

In [7]:
from langchain_core.messages import HumanMessage

human_message = """Github의 pull request를 확인하고 코드 리뷰를 작성해주세요.
코드 리뷰를 아래 기준으로 작성한 후에 당신의 의견을 PR에 comment로 달아주세요.

PR URL: https://github.com/oasishun/n8n_book/pull/2

기준:
1. 코드내 버그나 side effect이 없는지 
2. 코드내 보안 이슈가 없는지
3. 코드 가독성/중복/성능 이슈가 없는지

코드 리뷰 comment를 github에 남기고, 해당 내용을 코드관리담당자에게 아래 채널에 slack으로 멘션 해주세요.

channel_id: <#C043KM5156E>
코드관리담당자: <@U043KJB2ASX>
"""

stream_generator = agent.astream({"messages": human_message}, stream_mode="updates")



In [8]:
all_chunks = await process_stream(stream_generator)

if all_chunks:
    final_result = all_chunks[-1]
    print("\nFinal result", final_result)


Tool: pull_request_read
Tool Output: [{'type': 'text', 'text': 'diff --git a/.gitignore b/.gitignore\nindex cda22d1..c5a73ed 100644\n--- a/.gitignore\n+++ b/.gitignore\n@@ -3,6 +3,25 @@ node_modules/\n package-lock.json\n yarn.lock\n \n+# Python\n+__pycache__/\n+*.py[cod]\n+*$py.class\n+*.so\n+.Python\n+venv/\n+env/\n+ENV/\n+.venv\n+*.egg-info/\n+dist/\n+build/\n+*.egg\n+\n+# Jupyter Notebook\n+.ipynb_checkpoints\n+*.ipynb_checkpoints/\n+\n # Environment variables\n .env\n .env.local\n@@ -24,7 +43,5 @@ logs/\n .cursor/debug.log\n \n # Build outputs\n-dist/\n-build/\n *.tsbuildinfo\n \ndiff --git a/README.md b/README.md\nindex 230483c..e185aa3 100644\n--- a/README.md\n+++ b/README.md\n@@ -2,3 +2,33 @@\n \n n8n 학습 프로젝트입니다.\n \n+## Python Jupyter Notebook 환경\n+\n+### 환경 설정\n+\n+1. 가상환경 생성 및 활성화:\n+```bash\n+python3 -m venv venv\n+source venv/bin/activate  # macOS/Linux\n+```\n+\n+2. 패키지 설치:\n+```bash\n+pip install -r requirements.txt\n+```\n+\n+3. Jupyter Notebook 실행:\n+```bash\n+jupyter 